# Chapter 6 — Top-down Ontology Development
### Notebook 1 · Foundational ontologies

*Book reference: Section 6.1*

A foundational ontology is a set of very general categories agreed in advance, so that domain modellers make the same distinctions the same way. Its real product is not the categories — it is the **questions** that place things in them.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch06_toolkit as ch6
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

## 1. The categories

A DOLCE-flavoured tree, simplified to seven leaves. The `branch` column carries the oldest distinction in the subject: **endurants** exist *through* time (a giraffe is wholly present at every moment it exists), **perdurants** unfold *in* time (a hunt has temporal parts).

In [3]:
print(pd.DataFrame([{'id': c.id, 'name': c.name, 'branch': c.branch,
                     'examples': ', '.join(c.examples)}
                    for c in ch6.CATEGORIES]).to_string(index=False))

              id             name    branch                                    examples
 physical-object  Physical Object  Endurant                  a giraffe, a statue, a car
amount-of-matter Amount of Matter  Endurant                           clay, water, gold
         feature          Feature  Endurant                   a hole, a bump, a surface
         quality          Quality   Quality the colour of a leaf, a mass, a temperature
           event            Event Perdurant   a hunt that succeeds, an arrival, a birth
         process          Process Perdurant                 grazing, running, digestion
        abstract  Abstract Entity  Abstract      the number seven, a set, a proposition


In [4]:
for c in ch6.CATEGORIES:
    print(f'{c.name} ({c.branch})')
    print(f'   {c.gloss}')
    print(f'   e.g. {", ".join(c.examples)}\n')

Physical Object (Endurant)
   A bounded, countable thing that exists in time and has spatial parts.
   e.g. a giraffe, a statue, a car

Amount of Matter (Endurant)
   Stuff: any part of it is described by the same term (mass noun).
   e.g. clay, water, gold

Feature (Endurant)
   A dependent part or place of a host object.
   e.g. a hole, a bump, a surface

Quality (Quality)
   An aspect that inheres in a bearer and cannot exist alone.
   e.g. the colour of a leaf, a mass, a temperature

Event (Perdurant)
   Something that happens and has a natural culmination.
   e.g. a hunt that succeeds, an arrival, a birth

Process (Perdurant)
   Something that goes on without a built-in endpoint.
   e.g. grazing, running, digestion

Abstract Entity (Abstract)
   Outside space and time, and not dependent on a bearer.
   e.g. the number seven, a set, a proposition



## 2. The decision questions

This is the part that makes alignment an engineering activity. Each question is a yes/no test with a defensible answer for every category.

In [5]:
for key, question in ch6.DECISION_QUESTIONS.items():
    print(f'  {key:11s} {question}')

  happens     Does it happen or unfold in time, rather than merely exist through time?
  spatial     Does it have a location in space?
  mass        Is any part of it describable by the same term (is it a mass noun)?
  dependent   Must it inhere in, or belong to, some other entity to exist?
  telic       Does it have a natural endpoint or culmination?


In [6]:
rows = []
for c in ch6.CATEGORIES:
    row = {'category': c.name}
    row.update({k: ('yes' if v else 'no')
                for k, v in ch6.category_answers(c.id).items()})
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

        category happens spatial mass dependent telic
 Physical Object      no     yes   no        no    no
Amount of Matter      no     yes  yes        no    no
         Feature      no     yes   no       yes    no
         Quality      no      no   no       yes    no
           Event     yes     yes   no        no   yes
         Process     yes     yes   no        no    no
 Abstract Entity      no      no   no        no    no


## 3. Aligning a class by interrogation

`identify_category` takes whatever answers you have so far and returns the categories still consistent with them. Watch the candidate set shrink.

In [7]:
steps = [{}, {'happens': True}, {'happens': True, 'telic': True}]
for answers in steps:
    print(f'{str(answers):46s} -> {ch6.identify_category(answers)}')

{}                                             -> ['physical-object', 'amount-of-matter', 'feature', 'quality', 'event', 'process', 'abstract']
{'happens': True}                              -> ['event', 'process']
{'happens': True, 'telic': True}               -> ['event']


In [8]:
print('Aligning "clay":')
answers = {}
for question, answer in [('happens', False), ('spatial', True), ('mass', True)]:
    answers[question] = answer
    print(f'  {question}={answer}  -> candidates {ch6.identify_category(answers)}')
assert ch6.identify_category(answers) == ['amount-of-matter']

Aligning "clay":
  happens=False  -> candidates ['physical-object', 'amount-of-matter', 'feature', 'quality', 'abstract']
  spatial=True  -> candidates ['physical-object', 'amount-of-matter', 'feature']
  mass=True  -> candidates ['amount-of-matter']


In [9]:
print('Aligning "the colour of a leaf":')
answers = {}
for question, answer in [('happens', False), ('spatial', False), ('dependent', True)]:
    answers[question] = answer
    print(f'  {question}={answer}  -> candidates {ch6.identify_category(answers)}')
assert ch6.identify_category(answers) == ['quality']

Aligning "the colour of a leaf":
  happens=False  -> candidates ['physical-object', 'amount-of-matter', 'feature', 'quality', 'abstract']
  spatial=False  -> candidates ['quality', 'abstract']
  dependent=True  -> candidates ['quality']


> **Three questions, one answer.** That is what a foundational ontology buys: not a list of categories to memorise but a short, repeatable interrogation that two modellers will answer the same way. Notebook 4 hands the interrogation to an agent — and derives the optimal order of questions from a reward function.

## 4. DOLCE is not the only choice

BFO draws many of the same distinctions differently, and the mismatch is a real project decision rather than a detail.

In [10]:
print(pd.DataFrame(ch6.BFO_COMPARISON).to_string(index=False))

           dolce                                                 bfo                                           note
 Physical Object                            Material Entity (Object)                                  closest match
Amount of Matter        Material Entity (Object Aggregate / portion) BFO has no dedicated amount-of-matter category
         Feature Immaterial Entity (Site) / Continuant Fiat Boundary                         holes are sites in BFO
         Quality         Specifically Dependent Continuant (Quality)                                    close match
           Event                   Process (with a process boundary)                BFO folds events into processes
         Process                                             Process                                    close match
 Abstract Entity                             (none - BFO is realist)            BFO deliberately excludes abstracta


> The last row is the sharpest: BFO is **realist** and has no place for abstract entities at all. If your domain needs to talk about numbers, propositions or musical works as first-class things, that is not a preference — it decides the choice for you.

### Exercise 1.1 — Align four domain classes

Place `a hole in a leaf`, `digestion`, `the number seven` and `a herd` in the category tree by answering the decision questions. One of them exposes a limit of this seven-category tree — say which.

> **Hint.** Answer only the questions you need; `identify_category` accepts partial answers.

In [11]:
# YOUR CODE HERE


<details>
<summary>Solution 1.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [12]:
cases = {
    'a hole in a leaf': {'happens': False, 'spatial': True, 'mass': False,
                         'dependent': True},
    'digestion':        {'happens': True, 'telic': False},
    'the number seven': {'happens': False, 'spatial': False, 'dependent': False},
}
for name, answers in cases.items():
    print(f'{name:20s} -> {ch6.identify_category(answers)}')
assert ch6.identify_category(cases['a hole in a leaf']) == ['feature']
assert ch6.identify_category(cases['digestion']) == ['process']
assert ch6.identify_category(cases['the number seven']) == ['abstract']

herd = {'happens': False, 'spatial': True, 'mass': False, 'dependent': False}
print(f"{'a herd':20s} -> {ch6.identify_category(herd)}")
print('\nA herd comes out as a physical object, which is not wrong but is not\n'
      'informative either: this tree has no COLLECTION category, even though the\n'
      'part-whole taxonomy in Notebook 2 needs one for member-of. Real\n'
      'foundational ontologies do have it. The lesson is that your category set\n'
      'and your relation set have to be designed together.')

a hole in a leaf     -> ['feature']
digestion            -> ['process']
the number seven     -> ['abstract']
a herd               -> ['physical-object']

A herd comes out as a physical object, which is not wrong but is not
informative either: this tree has no COLLECTION category, even though the
part-whole taxonomy in Notebook 2 needs one for member-of. Real
foundational ontologies do have it. The lesson is that your category set
and your relation set have to be designed together.


### Exercise 1.2 — Is any question redundant?

Find the smallest set of questions that still distinguishes all seven categories. Then, for each question, name the pair of categories that *only* it separates.

In [13]:
# YOUR CODE HERE


<details>
<summary>Solution 1.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [14]:
import itertools
questions = list(ch6.DECISION_QUESTIONS)
best = None
for size in range(1, len(questions) + 1):
    for subset in itertools.combinations(questions, size):
        vectors = {c.id: tuple(ch6.category_answers(c.id)[q] for q in subset)
                   for c in ch6.CATEGORIES}
        if len(set(vectors.values())) == len(ch6.CATEGORIES):
            best = subset
            break
    if best:
        break
print('smallest sufficient question set:', best)
print('droppable:', [q for q in questions if q not in best] or 'none')
assert best is not None and len(best) == len(questions)

print('\nfor each question, the pair only it separates:')
for q in questions:
    others = [x for x in questions if x != q]
    collisions = {}
    for c in ch6.CATEGORIES:
        key = tuple(ch6.category_answers(c.id)[x] for x in others)
        collisions.setdefault(key, []).append(c.id)
    merged = [v for v in collisions.values() if len(v) > 1]
    print(f'  {q:11s} -> {merged}')
print('\nNo question is redundant: drop any one and two categories collapse into\n'
      'each other. That is a well-designed question set -- and it does NOT mean\n'
      'every alignment needs all five. Notebook 4 shows the optimal policy\n'
      'settling most classes in three, because a question is only asked on the\n'
      'branch where it still discriminates.')

smallest sufficient question set: ('happens', 'spatial', 'mass', 'dependent', 'telic')
droppable: none

for each question, the pair only it separates:
  happens     -> [['physical-object', 'process']]
  spatial     -> [['physical-object', 'abstract'], ['feature', 'quality']]
  mass        -> [['physical-object', 'amount-of-matter']]
  dependent   -> [['physical-object', 'feature'], ['quality', 'abstract']]
  telic       -> [['event', 'process']]

No question is redundant: drop any one and two categories collapse into
each other. That is a well-designed question set -- and it does NOT mean
every alignment needs all five. Notebook 4 shows the optimal policy
settling most classes in three, because a question is only asked on the
branch where it still discriminates.
